## Gradiente de SINGX

Para cada bloque $j=1,\dots,m$, con $y^{(j)}=(x_{4j-3},x_{4j-2},x_{4j-1},x_{4j})^t$, definimos

$$u_1^{(j)}=x_{4j-3}+10x_{4j-2},\quad u_2^{(j)}=x_{4j-1}-x_{4j},\quad u_3^{(j)}=x_{4j-2}-2x_{4j-1},\quad u_4^{(j)}=x_{4j-3}-x_{4j}.$$

Entonces

$$f(x)=\sum_{j=1}^{m}\Big[(u_1^{(j)})^2+5(u_2^{(j)})^2+(u_3^{(j)})^4+10(u_4^{(j)})^4\Big],$$

y aplicando la regla de la cadena se obtiene, para cada bloque,

$$\frac{\partial f}{\partial x_{4j-3}}=2u_1^{(j)}+40(u_4^{(j)})^3,\qquad \frac{\partial f}{\partial x_{4j-2}}=20u_1^{(j)}+4(u_3^{(j)})^3,$$

$$\frac{\partial f}{\partial x_{4j-1}}=10u_2^{(j)}-8(u_3^{(j)})^3,\qquad \frac{\partial f}{\partial x_{4j}}=-10u_2^{(j)}-40(u_4^{(j)})^3.$$

In [2]:
# Imports
import sys
from pathlib import Path

# Sube un nivel desde la carpeta del notebook hasta la raíz del repositorio
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
from funciones import singx_f, singx_grad, max_descenso

In [3]:
# Verificación rápida con n=4, x0=(1,1,1,1)
x0 = np.array([1.0, 1.0, 1.0, 1.0])
print("f(x0) =", singx_f(x0))          # debe dar 122.0
print("grad(x0) =", singx_grad(x0))    # debe dar [22, 216, 8, 0]
print("||grad(x0)|| =", np.linalg.norm(singx_grad(x0)))

f(x0) = 122.0
grad(x0) = [ 22. 216.   8.  -0.]
||grad(x0)|| = 217.26481537515457


## Método de Máximo Descenso 

El algoritmo se implementa numéricamente con una búsqueda lineal de Armijo. Dado $x_0\in\mathbb{R}^n$ y tolerancia $\varepsilon>0$:

1. Calcular $g_k=\nabla f(x_k)$. Si $\|g_k\|<\varepsilon$, parar.
2. Dirección: $d_k=-g_k/\|g_k\|$.
3. Búsqueda lineal: encontrar $\alpha_k$ que satisfaga Armijo
   $f(x_k+\alpha_k d_k)\le f(x_k)+c_1\alpha_k g_k^t d_k$.
4. Actualizar $x_{k+1}=x_k+\alpha_k d_k$, $k\leftarrow k+1$, volver al paso 1.

 La matriz Hessiana de SINGX es singular en el óptimo $x^*=0$ (rango $2m<n$), por lo que no se garantiza una razón de convergencia lineal acotada por $(\lambda-1)^2/(\lambda+1)^2$. En la práctica, el método requiere del orden de $10^3$–$10^4$ iteraciones para reducir el gradiente a $10^{-6}$, y la sucesión tiene el patrón de zigzag ($d_k^t d_{k+1}\approx 0$).

In [4]:
# Ejemplo con n=4, x0=(1,1,1,1): primera iteración 122 -> ~14.43
x0 = np.array([1., 1., 1., 1.])
print(f"f(x0) = {singx_f(x0):.4f}")
x_opt, hist = max_descenso(singx_f, singx_grad, x0)

print(f"\nf(x_opt)  = {singx_f(x_opt):.6e}")
print(f"x_opt     = {x_opt}")
print(f"Iteraciones ejecutadas: {len(hist['f'])-1}")
print(f"f tras 1ª iteración: {hist['f'][1]:.4f}")

f(x0) = 122.0000
k=   0  f=1.452831e+01  ||g||=5.708e+01  alpha=1.000e+00
k=   1  f=1.159343e+01  ||g||=5.300e+01  alpha=1.000e+00
k=   2  f=8.712969e+00  ||g||=4.789e+01  alpha=5.000e-01
k=   3  f=2.772921e+00  ||g||=1.071e+01  alpha=2.500e-01
k=   4  f=1.306090e+00  ||g||=1.239e+01  alpha=2.500e-01
k= 100  f=6.788734e-03  ||g||=7.933e-01  alpha=7.812e-03
k= 200  f=3.865595e-03  ||g||=7.915e-01  alpha=7.812e-03
k= 300  f=2.830556e-03  ||g||=7.909e-01  alpha=7.812e-03
k= 400  f=2.355088e-03  ||g||=7.905e-01  alpha=7.812e-03
k= 500  f=4.128494e-06  ||g||=7.706e-04  alpha=7.629e-06
k= 600  f=4.004475e-06  ||g||=7.706e-04  alpha=7.629e-06
k= 700  f=3.885977e-06  ||g||=7.706e-04  alpha=7.629e-06
k= 800  f=3.772672e-06  ||g||=7.706e-04  alpha=7.629e-06
k= 900  f=3.664261e-06  ||g||=7.706e-04  alpha=7.629e-06
k=1000  f=3.560464e-06  ||g||=7.706e-04  alpha=7.629e-06
k=1100  f=3.461022e-06  ||g||=7.706e-04  alpha=7.629e-06
k=1200  f=3.365693e-06  ||g||=7.706e-04  alpha=7.629e-06
k=1300  f=3.27